# Vertical temperature profiles on the native cubed-sphere mesh

Time-mean temperature-pressure profiles of an HD 209458b simulation, taken from the UGRID output
(`lfric_diag_native*.nc`) without regridding:
- profiles in the mesh cells nearest to a selection of points along a given latitude;
- optionally, the day-side and/or night-side area-weighted mean, computed with `aeolus.lfric.ugrid_spatial`;
- the initial profile from `lfric_initial.nc`.

In [ ]:
import warnings

import cartopy.crs as ccrs
from cmcrameri import cm
import iris
import matplotlib.pyplot as plt
import numpy as np
import paths
import seaborn as sns
from aeolus.calc import time_mean
from aeolus.lfric import load_lfric_raw, ugrid_spatial
from aeolus.model import lfric
from aeolus.plot import figsave, unit_format
from shapely.geometry import Polygon
from common import (
    EXPERIMENTS,
    PA_TO_BAR,
    load_exp_days,
    setup_pressure_axis,
)

warnings.filterwarnings("ignore")
iris.FUTURE.date_microseconds = True

print(*[k for k in EXPERIMENTS if k.startswith("hd209")], sep="\n")

## Functions for UGRID cubes

In [ ]:
def _lonlat_to_xyz(lons, lats):
    """Convert longitudes and latitudes (degrees) to unit vectors."""
    lam, phi = np.deg2rad(np.asarray(lons, float)), np.deg2rad(np.asarray(lats, float))
    return np.stack(
        [np.cos(phi) * np.cos(lam), np.cos(phi) * np.sin(lam), np.sin(phi)], axis=-1
    )


def nearest_face_indices(cube, lons, lats):
    """
    Find the mesh faces with centres nearest (in great-circle distance) to the given points.

    Parameters
    ----------
    cube: iris.cube.Cube
        Cube located on mesh faces.
    lons, lats: array-like
        Longitudes and latitudes of the points (degrees).

    Returns
    -------
    numpy.ndarray of face indices, one per point.
    """
    face_xyz = _lonlat_to_xyz(cube.coord(lfric.x).points, cube.coord(lfric.y).points)
    pts_xyz = _lonlat_to_xyz(
        *np.broadcast_arrays(np.asarray(lons, float), np.asarray(lats, float))
    )
    # The nearest face has the largest dot product (cosine of the distance)
    return np.argmax(pts_xyz @ face_xyz.T, axis=-1)


def mesh_face_areas(cube, radius=1.0):
    """
    Areas of the (convex) faces of the mesh a cube is located on.

    Each face is split into triangles around its first node, and the area
    of each spherical triangle is found from its solid angle
    (Van Oosterom & Strackee, 1983).
    """
    mesh = cube.mesh
    face_nodes = np.asarray(mesh.face_node_connectivity.indices_by_location())
    node_xyz = _lonlat_to_xyz(
        mesh.node_coords.node_x.points, mesh.node_coords.node_y.points
    )
    corners = node_xyz[face_nodes]  # (n_faces, n_corners, 3)
    area = np.zeros(face_nodes.shape[0])
    a = corners[:, 0]
    for j in range(1, face_nodes.shape[1] - 1):
        b, c = corners[:, j], corners[:, j + 1]
        numer = np.abs(np.einsum("ij,ij->i", a, np.cross(b, c)))
        denom = (
            1
            + np.einsum("ij,ij->i", a, b)
            + np.einsum("ij,ij->i", b, c)
            + np.einsum("ij,ij->i", c, a)
        )
        area += 2 * np.arctan2(numer, denom)
    return area * radius**2


def hemisphere_mask(cube, side, substellar_lon=0.0):
    """
    Boolean mask of mesh faces on the day side or night side of a tidally locked planet.

    A face is on the day side if the cosine of the stellar zenith angle
    at its centre is positive, i.e. its longitude is within 90 degrees
    of the substellar longitude.
    """
    dlon = np.deg2rad(cube.coord(lfric.x).points - substellar_lon)
    if side == "day":
        return np.cos(dlon) > 0
    elif side == "night":
        return np.cos(dlon) < 0
    raise ValueError(f"side must be 'day' or 'night', got {side!r}")


def ugrid_hemisphere_mean(cube, side, substellar_lon=0.0, areas=None):
    """
    Area-weighted mean of a UGRID cube over the day-side or night-side hemisphere.

    The mean is over all mesh faces using `aeolus.lfric.ugrid_spatial`,
    with zero weights for the faces of the opposite hemisphere.
    """
    if areas is None:
        areas = mesh_face_areas(cube)
    weights = areas * hemisphere_mask(cube, side, substellar_lon=substellar_lon)
    (mesh_dim,) = cube.coord_dims(lfric.x)
    shape = [1] * cube.ndim
    shape[mesh_dim] = -1
    weights = np.broadcast_to(weights.reshape(shape), cube.shape)
    return ugrid_spatial(cube, "mean", weights=weights)


def initial_temperature_profile(ds_init, const, face_idx=0):
    """
    Temperature and pressure profiles in one column of `lfric_initial.nc`.

    The temperature is T = theta * Pi and the pressure is p = p_zero * Pi**(1 / kappa),
    where theta is the potential temperature, Pi is the Exner pressure,
    p_zero is the reference pressure and kappa = R / c_p, both from `const`.

    The initial state has the potential temperature on cell interfaces (full levels)
    and the Exner pressure in cell centres (half levels), so the potential temperature
    is interpolated linearly in height to the cell centres.
    The initial state of the RT simulations is horizontally uniform,
    so any column will do.
    """
    theta = ds_init.extract_cube("air_potential_temperature")[:, face_idx]
    exner = ds_init.extract_cube("exner_pressure")[:, face_idx]
    z_full = ds_init.extract_cube("height_of_cell_interfaces_above_earth_radius")[
        :, face_idx
    ]
    z_half = ds_init.extract_cube("height_of_cell_centres_above_earth_radius")[
        :, face_idx
    ]
    theta_half = np.interp(z_half.data, z_full.data, theta.data)
    kappa = (const.dry_air_gas_constant / const.dry_air_spec_heat_press).data
    p_zero = const.reference_surface_pressure.copy()
    p_zero.convert_units("Pa")
    temp = exner.copy(data=theta_half * exner.data)
    temp.rename("air_temperature")
    temp.units = "K"
    pres = exner.copy(data=p_zero.data * exner.data ** (1 / kappa))
    pres.rename("air_pressure")
    pres.units = "Pa"
    return temp, pres

## Parameters

In [ ]:
EXP_KEY = "hd209_rand_t100k_nf20_tau1e5_amp1e-2_sclht6"
N_DAYS_MEAN = (
    200  # length of the averaging period (native output is written every 10 days)
)
LATITUDE = 0.0  # latitude of the profiles (deg)
# Longitudes of the profiles (deg) relative to the substellar point, positive eastward
# (labelled E/W in the plot): 0 = substellar point, +90 = eastern terminator (90E),
# -90 = western terminator (90W),
# -180 (same as +180) = antistellar point
REL_LONGITUDES = np.arange(-180, 180, 90)
# Hemisphere mean(s) to add: None, "day", "night" or ("day", "night")
HEMISPHERE_MEAN = ("day", "night")
PRES_LIM = (
    None  # (bottom, top) of the plot in bar, e.g. (200, 1e-5); None for the full range
)
SAVE_FIGS = True

EXP = EXPERIMENTS[EXP_KEY]
END_DAY = EXP.run_length
START_DAY = END_DAY - N_DAYS_MEAN
PERIOD_STR = f"days {START_DAY}-{END_DAY}" if N_DAYS_MEAN > 10 else f"day {END_DAY}"
# Wrap the relative longitudes to [-180, 180) and sort them
REL_LONGITUDES = np.unique((np.asarray(REL_LONGITUDES) + 180) % 360 - 180)
if isinstance(HEMISPHERE_MEAN, str):
    HEMISPHERE_MEAN = (HEMISPHERE_MEAN,)
elif HEMISPHERE_MEAN is None:
    HEMISPHERE_MEAN = ()
print(
    EXP.title, EXP.label, PERIOD_STR, f"substellar lon: {EXP.substellar_lon}", sep="\n"
)

## Load the data

In [ ]:
ds = load_exp_days(EXP_KEY, START_DAY, END_DAY, file_glob="lfric_diag_native*.nc")
temp = time_mean(ds.extract_cube("air_temperature"))
pres = time_mean(ds.extract_cube("air_pressure"))
# Temperature and pressure are on the same (full) levels in the native output
assert temp.coord(lfric.z) == pres.coord(lfric.z)
print(temp.summary(shorten=True))

ds_init = load_lfric_raw(
    paths.data_work / "lfric" / EXP.group / EXP.label / "lfric_initial.nc"
)
temp_init, pres_init = initial_temperature_profile(ds_init, EXP.const)

In [ ]:
# Points along the latitude, and the nearest mesh faces
lons = (EXP.substellar_lon + REL_LONGITUDES) % 360
face_idx = nearest_face_indices(temp, lons, LATITUDE)
for rel_lon, i in zip(REL_LONGITUDES, face_idx):
    print(
        f"{rel_lon:5.0f} -> face {i:5d} at"
        f" ({temp.coord(lfric.x).points[i]:7.2f}, {temp.coord(lfric.y).points[i]:6.2f})"
    )

# Hemisphere means
areas = mesh_face_areas(temp)
print(f"Sum of face areas / 4pi = {areas.sum() / (4 * np.pi):.6f}")
hemi_means = {
    side: (
        ugrid_hemisphere_mean(temp, side, EXP.substellar_lon, areas=areas),
        ugrid_hemisphere_mean(pres, side, EXP.substellar_lon, areas=areas),
    )
    for side in HEMISPHERE_MEAN
}

## Plot the profiles

Coloured lines are the profiles at the selected points, labelled by their longitude relative
to the substellar point (E/W of the substellar point, so 90E is the eastern terminator and 90W the western terminator). Thick lines are the hemisphere means, the dashed black line is the initial state.
The day-side and night-side means are averages of the temperature and of the pressure
in each model layer, i.e. on surfaces of constant height.
The inset map shows the locations of the profiles (markers), and the day and night sides (red and blue shading); the map is centred on the substellar point.


In [ ]:
KW_HEMI = {
    "day": {"color": "tab:red", "lw": 4, "alpha": 0.5, "label": "Day-side mean"},
    "night": {"color": "tab:blue", "lw": 4, "alpha": 0.5, "label": "Night-side mean"},
}


def add_location_map(
    ax, colors, bounds=(0.0, 0.0, 0.45, 0.3), projection_name="EqualEarth"
):
    """
    Add a small map in the bottom left corner of `ax` showing the locations of the profiles.

    The map is centred on the substellar point, and the day and night sides are shaded
    in the colours of the hemisphere means (`KW_HEMI`).
    """
    axm = ax.inset_axes(
        bounds,
        projection=getattr(ccrs, projection_name, "EqualEarth")(
            central_longitude=EXP.substellar_lon
        ),
    )
    axm.set_global()
    # Shade the day side and the night side (longitudes within 90 degrees of the
    # substellar and antistellar points) in the colours of the hemisphere means;
    # the outlines are densely sampled, otherwise their edges along the poles
    # collapse when projected
    n = 91
    lat = np.linspace(-90, 90, n)
    for side, centre in [("day", 0), ("night", 180)]:
        lon = np.linspace(-90, 90, n) + EXP.substellar_lon + centre
        hemi = Polygon(
            np.concatenate([
                np.column_stack([lon, np.full(n, -90)]),
                np.column_stack([np.full(n, lon[-1]), lat]),
                np.column_stack([lon[::-1], np.full(n, 90)]),
                np.column_stack([np.full(n, lon[0]), lat[::-1]]),
            ])
        )
        axm.add_geometries(
            [hemi],
            crs=ccrs.PlateCarree(),
            facecolor=KW_HEMI[side]["color"],
            alpha=0.25,
            edgecolor="none",
        )
    axm.gridlines(
        xlocs=np.arange(-180, 181, 60),
        ylocs=np.arange(-90, 91, 30),
        linewidth=0.3,
        color="0.5",
        # draw_labels=True,
    )
    axm.scatter(
        temp.coord(lfric.x).points[face_idx],
        temp.coord(lfric.y).points[face_idx],
        c=colors,
        s=2 ** (6 - np.log10(len(face_idx))),
        edgecolor="w",
        linewidth=0.75,
        transform=ccrs.PlateCarree(),
        zorder=3,
        clip_on=False,  # show markers at the map edge in full
    )
    axm.spines["geo"].set_linewidth(0.5)
    return axm


def lat_str(lat, tex=True):
    """Latitude as e.g. 30N, 30S or 0 (with a degree sign if `tex`)."""
    deg = "$^\\circ$" if tex else ""
    hemi = "" if np.isclose(lat, 0) else ("N" if lat > 0 else "S")
    return f"{abs(lat):.0f}{deg}{hemi}"


def lon_str(lon, tex=True):
    """Longitude in [-180, 180) as e.g. 45E, 45W, 0 or 180 (with a degree sign if `tex`)."""
    lon = (lon + 180) % 360 - 180
    deg = "$^\\circ$" if tex else ""
    hemi = (
        "" if np.isclose(lon, 0) or np.isclose(lon, -180) else ("E" if lon > 0 else "W")
    )
    return f"{abs(lon):.0f}{deg}{hemi}"


def rel_lon_colors(rel_lons, cmap):
    """Colours of the profiles from a colour map, fixed by the relative longitude."""
    return plt.get_cmap(cmap)((np.asarray(rel_lons) % 360) / 360)


def plot_temp_profiles(pres_lim=PRES_LIM, cmap="twilight_shifted_r"):
    fig, ax = plt.subplots(figsize=(4, 4.5), layout="constrained")
    colors = rel_lon_colors(REL_LONGITUDES, cmap=cmap)
    for rel_lon, i, color in zip(REL_LONGITUDES, face_idx, colors):
        ax.plot(
            temp.data[:, i],
            pres.data[:, i] * PA_TO_BAR,
            color=color,
            lw=1.25,
            label=lon_str(rel_lon),
        )
    for side, (t_mean, p_mean) in hemi_means.items():
        ax.plot(t_mean.data, p_mean.data * PA_TO_BAR, **KW_HEMI[side])
    ax.plot(
        temp_init.data,
        pres_init.data * PA_TO_BAR,
        color="k",
        lw=1.25,
        ls="--",
        dash_capstyle="round",
        label="Initial",
    )
    setup_pressure_axis(ax, lim=pres_lim)
    ax.set_xlabel("Temperature [K]")
    sns.despine(ax=ax, offset=5)
    ax.legend(
        title="Longitude from\nsubstellar point",
        loc="center left",
        fontsize="small",
        title_fontsize="small",
        frameon=False,
    )
    add_location_map(ax, colors)
    ax.set_title(
        f"{EXP.title} ({EXP.label})\nLatitude: {lat_str(LATITUDE)} | {PERIOD_STR}",
        size="small",
    )
    if SAVE_FIGS:
        figsave(
            fig,
            paths.drafts
            / "profiles"
            / f"{EXP_KEY}__temp_profiles__lat_{lat_str(LATITUDE, tex=False)}__days_{START_DAY}_{END_DAY}",
        )
    return fig


plot_temp_profiles(pres_lim=(200, 1e-4));